# Chapter 3 · 설비 이상·품질 예측

**연구질문:** 과거 센서 요약통계로 향후 12시간 내 고장을 예측할 수 있는가? 누수 없이 평가하면 성능은?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**지도학습 형태의 예측:** $\hat y_{t}=f(y_{t-1},\dots,y_{t-L},\ \text{rolling}_{t-1},\ \text{calendar}_t,\ x_t)$. 모든 특성은 **예측 시점 $t$에 알 수 있는 정보**만 사용한다(롤링 통계는 반드시 한 칸 이동).

**시간순 검증:** 무작위 분할 대신 훈련 구간이 항상 검증 구간보다 앞서는 expanding-window 분할을 쓴다.

**기준선 대비 성능:** $\mathrm{MASE}<1$이면 1단계 나이브보다 낫다. 수익률처럼 예측력이 약한 자료는 **표본 외 $R^2_{OS}=1-\dfrac{\sum(y_t-\hat y_t)^2}{\sum(y_t-\bar y_{t-1})^2}$** ($\bar y_{t-1}$: 직전까지의 평균)로 평가한다.

**변수 중요도:** 순열 중요도는 특성을 섞었을 때 검증 오차가 늘어난 정도이며 인과적 중요도가 아니다. 상관된 특성이 있으면 해석이 불안정하다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def synthetic_sensor(n_days=180, seed=SEED, n_faults=14):
    # NOT real data: 일주기 + 느린 드리프트 + 잡음, 고장 직전 12시간 동안 진동이 증가하는 전조 구간 포함
    rng = np.random.default_rng(seed); n = n_days * 24; t = np.arange(n)
    vib = 1.0 + 0.15 * np.sin(2 * np.pi * (t % 24) / 24) + 0.0005 * t + rng.normal(0, 0.05, n)
    temp = 60 + 3 * np.sin(2 * np.pi * (t % 24) / 24 - 0.5) + rng.normal(0, 0.6, n)
    fault = np.zeros(n, int)
    starts = np.sort(rng.choice(np.arange(72, n - 24), n_faults, replace=False))
    for s0 in starts:
        for k in range(12):
            vib[s0 - 12 + k] += 0.03 * (k + 1); temp[s0 - 12 + k] += 0.15 * (k + 1)
        vib[s0:s0 + 2] += 0.8; fault[s0] = 1
    idx = pd.date_range("2024-01-01", periods=n, freq="h")
    return pd.DataFrame({"vibration": vib, "temp": temp, "fault": fault}, index=idx)

def synthetic_demand(n=1095, seed=SEED + 2):
    rng = np.random.default_rng(seed); t = np.arange(n)
    mu = 500 + 0.15 * t + 40 * np.sin(2 * np.pi * t / 365.25) + 60 * (t % 7 < 5) + rng.normal(0, 12, n)
    return pd.Series(mu, index=pd.date_range("2021-01-01", periods=n, freq="D"), name="demand")

def load_manufacturing(which="sensor"):
    # 달력 시간이 있는 무료 제조 시계열은 드물다(카탈로그 3절). 기본은 합성 자료이며 NOT real data로 표시한다.
    if which == "demand":
        return dict(y=synthetic_demand(), m=7, freq="D", label="일별 제조 수요(합성)", unit="개",
                    kind="SYNTHETIC (NOT real data)")
    df = synthetic_sensor()
    return dict(y=df["vibration"], df=df, m=24, freq="h", label="진동 센서(합성)", unit="mm/s",
                kind="SYNTHETIC (NOT real data)")

D = load_manufacturing("sensor")
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min(), "~", y.index.max())

## 2. 특성 구성·모형·평가

In [ ]:
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              RandomForestClassifier, HistGradientBoostingClassifier)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

def calendar_features(idx, freq):
    out = pd.DataFrame(index=idx)
    if freq in ("D", "B", "h"):
        out["dow"] = idx.dayofweek
    if freq == "h":
        out["hour"] = idx.hour
    out["month_sin"] = np.sin(2 * np.pi * idx.month / 12); out["month_cos"] = np.cos(2 * np.pi * idx.month / 12)
    return out

def lag_features(s, lags, rolls=()):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rmean_{w}"] = s.shift(1).rolling(w).mean()          # shift(1): 현재 값을 쓰지 않는다(누수 방지)
        X[f"rstd_{w}"] = s.shift(1).rolling(w).std()
    return X

def get_models():
    models = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.6, n_jobs=-1, random_state=SEED),
        "HistGB": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=SEED),
    }
    if ensure("xgboost"):                                         # 선택: 설치돼 있으면 XGBoost도 비교
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                                         random_state=SEED, n_jobs=-1, verbosity=0)
    return models

def fit_eval(X, target, n_test, models=None):
    # 앞쪽으로 학습, 마지막 n_test 관측으로 평가 (1단계 예측, 시차 입력은 실제값)
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    tr, te = data.iloc[:-n_test], data.iloc[-n_test:]
    Xtr, ytr, Xte, yte = tr.drop(columns="target"), tr["target"], te.drop(columns="target"), te["target"]
    models = models or get_models(); preds = {}
    for name, mdl in models.items():
        mdl.fit(Xtr, ytr); preds[name] = pd.Series(mdl.predict(Xte), index=te.index)
    return dict(Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, preds=preds, models=models, data=data)

def cv_rmse(X, target, models, n_splits=4, test_size=None):
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    test_size = test_size or max(30, len(data) // 10)
    tss = TimeSeriesSplit(n_splits=n_splits, test_size=test_size); out = {k: [] for k in models}
    from sklearn.base import clone
    for tr_i, te_i in tss.split(data):
        a, b = data.iloc[tr_i], data.iloc[te_i]
        for name, mdl in models.items():
            m_ = clone(mdl).fit(a.drop(columns="target"), a["target"])
            out[name].append(rmse(b["target"], m_.predict(b.drop(columns="target"))))
    return pd.DataFrame(out).agg(["mean", "std"]).T.rename(columns={"mean": "CV_RMSE_mean", "std": "CV_RMSE_sd"})

In [ ]:
# 결과표: 기준선과 모델, 마지막 구간 평가 + 시간순 CV
def report(res, baselines, train_for_scale):
    rows = {}
    for name, f in {**baselines, **res["preds"]}.items():
        rows[name] = {"MAE": mae(res["yte"], f), "RMSE": rmse(res["yte"], f),
                      "MASE": mase(res["yte"], f, train_for_scale, m=1)}
    tab = pd.DataFrame(rows).T.sort_values("RMSE"); show(tab); return tab

In [ ]:
df_m = D["df"].copy(); HOR = 12                                       # 향후 12시간 내 고장 여부
fut = pd.concat([df_m["fault"].shift(-k) for k in range(1, HOR + 1)], axis=1).max(axis=1)
target = fut.fillna(0).astype(int).rename("fault_next12")
def roll_feats(s, name):
    f = pd.DataFrame(index=s.index)
    for w in (6, 12, 24):
        f[f"{name}_mean{w}"] = s.rolling(w).mean()             # 현재 시점까지의 과거 창만 사용
        f[f"{name}_std{w}"] = s.rolling(w).std()
    f[f"{name}_slope12"] = (s - s.shift(12)) / 12
    return f
X = pd.concat([roll_feats(df_m["vibration"], "vib"), roll_feats(df_m["temp"], "temp")], axis=1)
data = pd.concat([X, target], axis=1).dropna()
n_tr = int(len(data) * 0.7)
tr, te = data.iloc[:n_tr], data.iloc[n_tr:]
Xtr, ytr, Xte, yte = tr.drop(columns="fault_next12"), tr["fault_next12"], te.drop(columns="fault_next12"), te["fault_next12"]
print(f"학습 {len(tr)}시간(양성 비율 {ytr.mean():.3f}) | 검증 {len(te)}시간(양성 비율 {yte.mean():.3f})")
print("검증구간 고장 이벤트 수(참고):", int(D['df'].loc[te.index, 'fault'].sum()), "-> 이벤트가 적으면 지표가 불안정")
clfs = {
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "RandomForest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED),
}
rows, probs = [], {}
for name, c in clfs.items():
    c.fit(Xtr, ytr); p = c.predict_proba(Xte)[:, 1]; probs[name] = p
    rows.append({"model": name, "ROC_AUC": roc_auc_score(yte, p), "PR_AUC": average_precision_score(yte, p)})
rows.append({"model": "무작위(PR-AUC=양성비율)", "ROC_AUC": 0.5, "PR_AUC": float(yte.mean())})
show(pd.DataFrame(rows).set_index("model"))

In [ ]:
# 임계값은 학습구간에서만 정하고(재현율 80% 목표) 검증구간에 그대로 적용한다
best_name = max(probs, key=lambda k: average_precision_score(yte, probs[k]))
p_tr = clfs[best_name].predict_proba(Xtr)[:, 1]
prec, rec, thr = precision_recall_curve(ytr, p_tr)
ok = np.where(rec[:-1] >= 0.8)[0]; th = thr[ok[-1]] if len(ok) else 0.5
pred = (probs[best_name] >= th).astype(int)
tp = int(((pred == 1) & (yte.to_numpy() == 1)).sum()); fp = int(((pred == 1) & (yte.to_numpy() == 0)).sum())
fn = int(((pred == 0) & (yte.to_numpy() == 1)).sum())
print(f"모델={best_name} | 임계값(학습구간 기준)={th:.3f} | TP={tp} FP={fp} FN={fn} | "
      f"정밀도={tp/max(tp+fp,1):.3f} 재현율={tp/max(tp+fn,1):.3f}")
imp = pd.Series(permutation_importance(clfs[best_name], Xte, yte, n_repeats=3, random_state=SEED,
                scoring="average_precision").importances_mean, index=Xte.columns).sort_values(ascending=False)
show(imp.head(6).to_frame("PR-AUC 감소량"))

**읽는 법:** 합성 센서에는 고장 직전 12시간의 전조가 **심어져 있으므로** 높은 성능은 방법이 작동한다는 점검일 뿐이며 현장 성능이 아니다. 실제 설비 자료로는 (i) 고장 라벨의 시점 정의, (ii) 이벤트 수, (iii) 경보 후 조치 비용이 평가를 좌우한다. 평가 누수는 Ch.7에서 다룬다.

## 3. 결과 해석 작성
1. 기준선(나이브·계절 나이브·역사평균)보다 나은 모델이 있는가? 개선폭은 CV 변동성($sd$) 대비 큰가?
2. 변수 중요도가 도메인 지식과 일관되는가? (인과적 해석 금지)
3. 가장 위험한 누수 경로 3개를 적고, 이 노트북에서 어떻게 막았는지(롤링 shift, 학습구간 통계 스케일, 시간순 분할)를 서술한다.
4. `DATA_KIND`가 합성이면 결과는 **방법 점검**이다. 실제 자료로 바꿨을 때 달라질 점을 적는다.

**제출:** 결과표(기준선 포함), CV 표, 중요도 그래프/표, 해석 단락, 한계.